# Innovate Library: Comprehensive Usage Guide

This notebook demonstrates the key features of the innovate library for modeling innovation and policy diffusion.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from innovate.diffuse.bass import BassModel
from innovate.fitters.scipy_fitter import ScipyFitter

## 1. Basic Bass Model Fitting

Let's start with fitting a simple Bass model to synthetic data.

In [ ]:
# Generate some synthetic data using known parameters
true_p = 0.03  # coefficient of innovation
true_q = 0.35  # coefficient of imitation
true_m = 1000  # market potential

# Generate time points
t_data = np.linspace(0, 10, 50)

# Generate cumulative adoption using the Bass model equation
# The continuous solution: F(t) = m * (1 - exp(-(p+q)*t)) / (1 + (q/p)*exp(-(p+q)*t))
cumulative_adoption = true_m * (1 - np.exp(-(true_p + true_q) * t_data)) / (1 + (true_q / true_p) * np.exp(-(true_p + true_q) * t_data))

# Add some noise to make it more realistic
noisy_adoption = cumulative_adoption + np.random.normal(0, 20, size=cumulative_adoption.shape)
noisy_adoption = np.maximum(noisy_adoption, 0)  # Ensure non-negative values

print(f"Generated {len(t_data)} data points")
print(f"Adoption ranges from {noisy_adoption.min():.1f} to {noisy_adoption.max():.1f}")

In [ ]:
# Create model and fitter
model = BassModel()
fitter = ScipyFitter()

# Fit the model to the data
fitted_model = model.fit(fitter, t_data, noisy_adoption)

# Print the fitted parameters
print("Fitted Parameters:")
for param_name, param_value in fitted_model.params_.items():
    print(f"  {param_name}: {param_value:.4f}")

# Get the R² score
r_squared = fitted_model.score(t_data, noisy_adoption)
print(f"\nR² Score: {r_squared:.4f}")

In [ ]:
# Generate predictions
t_pred = np.linspace(0, 12, 100)
predictions = fitted_model.predict(t_pred)

# Plot results
plt.figure(figsize=(12, 6))
plt.plot(t_data, noisy_adoption, 'bo', label='Observed Data', markersize=6)
plt.plot(t_pred, predictions, 'r-', label='Bass Model Fit', linewidth=2)
plt.xlabel('Time')
plt.ylabel('Cumulative Adoption')
plt.title('Bass Model: Fitting Innovation Diffusion Data')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. Using Covariates

The innovate library allows you to include external variables (covariates) that affect model parameters.

In [ ]:
# Create synthetic data with covariate effects
time_points = np.linspace(0, 10, 30)

# Simulate marketing spend over time
marketing_spend = 100 * np.exp(-0.3 * time_points) + 20

# Generate cumulative adoption with marketing effect
base_p = 0.02
base_q = 0.25
base_m = 800
marketing_effect = 0.0001  # How marketing spend affects innovation coefficient

# For simplicity, we'll create data that approximates the effect
adoption_with_marketing = np.zeros(len(time_points))
current_adoption = 0

for i in range(len(time_points)):
    if i == 0:
        dt = 1
    else:
        dt = time_points[i] - time_points[i-1]
    
    # Effective parameters changed by marketing spend
    effective_p = base_p + marketing_effect * marketing_spend[i]
    
    # Calculate adoption change
    adoption_change = (effective_p + base_q * (current_adoption / base_m)) * (base_m - current_adoption) * dt
    current_adoption += adoption_change
    
    # Add some noise
    adoption_with_marketing[i] = current_adoption + np.random.normal(0, 5)

# Ensure non-negative
adoption_with_marketing = np.maximum(adoption_with_marketing, 0)

# Plot the data and marketing spend
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

ax1.plot(time_points, adoption_with_marketing, 'bo-', label='Adoption with Marketing')
ax1.set_xlabel('Time')
ax1.set_ylabel('Cumulative Adoption')
ax1.set_title('Adoption with Marketing Covariate')
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2.plot(time_points, marketing_spend, 'go-', label='Marketing Spend')
ax2.set_xlabel('Time')
ax2.set_ylabel('Marketing Spend')
ax2.set_title('Marketing Spend Over Time')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Create a Bass model with covariates
model_with_covariates = BassModel(covariates=["marketing_spend"])
fitter = ScipyFitter()

# Create the covariates dictionary
covariates_dict = {"marketing_spend": marketing_spend}

# Fit the model to the data with covariates
fitted_model_with_covariates = model_with_covariates.fit(
    fitter, 
    time_points, 
    adoption_with_marketing
)

# Print the fitted parameters (including covariate effects)
print("Fitted Parameters with Covariates:")
for param_name, param_value in fitted_model_with_covariates.params_.items():
    print(f"  {param_name}: {param_value:.4f}")

# Get the R² score
r_squared_with_cov = fitted_model_with_covariates.score(time_points, adoption_with_marketing, covariates=covariates_dict)
print(f"\nR² Score with Covariates: {r_squared_with_cov:.4f}")

In [ ]:
# Generate predictions with covariates
t_pred_full = np.linspace(0, 12, 100)

# Extend marketing spend to prediction period using exponential decay
extended_marketing = 100 * np.exp(-0.3 * t_pred_full) + 20
extended_covariates = {"marketing_spend": extended_marketing}

predictions_with_covariates = fitted_model_with_covariates.predict(
    t_pred_full, 
    covariates=extended_covariates
)

# Also predict without covariates for comparison
predictions_without_covariates = fitted_model_with_covariates.predict(t_pred_full)

# Plot results
plt.figure(figsize=(12, 6))
plt.plot(time_points, adoption_with_marketing, 'bo', label='Observed Data', markersize=6)
plt.plot(t_pred_full, predictions_with_covariates, 'r-', label='Model with Marketing Covariate', linewidth=2)
plt.plot(t_pred_full, predictions_without_covariates, 'g--', label='Model without Marketing Covariate', linewidth=2)
plt.xlabel('Time')
plt.ylabel('Cumulative Adoption')
plt.title('Bass Model: Effect of Marketing Covariate')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Predicting Adoption Rate

The library also provides methods to predict the rate of adoption (new adoptions per unit of time).

In [ ]:
# Predict adoption rates
adoption_rates = fitted_model.predict_adoption_rate(t_pred)
adoption_rates_with_covariates = fitted_model_with_covariates.predict_adoption_rate(
    t_pred_full, 
    covariates=extended_covariates
)

# Plot adoption rates
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

ax1.plot(t_pred, adoption_rates, 'r-', label='Adoption Rate (Simple Model)', linewidth=2)
ax1.set_xlabel('Time')
ax1.set_ylabel('Adoption Rate')
ax1.set_title('Rate of New Adoptions Over Time (Simple Model)')
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2.plot(t_pred_full, adoption_rates_with_covariates, 'b-', label='Adoption Rate (with Marketing)', linewidth=2)
ax2.set_xlabel('Time')
ax2.set_ylabel('Adoption Rate')
ax2.set_title('Rate of New Adoptions Over Time (with Marketing Covariate)')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

## 4. Comparing with Other Models

The innovate library contains multiple diffusion models. Let's compare the Bass model with others.

In [ ]:
# For now, let's just show the different model types available
print("Available Models in Innovate Library:")
print("- Diffusion Models:")
print("  - BassModel: Classic innovation-imitation model")
print("  - LogisticModel: S-shaped growth model")
print("  - GompertzModel: Growth model with exponentially decaying growth rate")
print()
print("- Substitution Models:")
print("  - Fisher-Pry Model: For technology substitution")
print("  - Norton-Bass Model: For competitive diffusion")
print()
print("- Competition Models:")
print("  - MultiProductDiffusionModel: For competing innovations")
print()
print("- Hype Models:")
print("  - For simulating Gartner Hype Cycle")
print()
print("- Dynamics Models:")
print("  - Contagion Models (SIR, SIS, SEIR)")
print("  - Competition Dynamics (Lotka-Volterra, Replicator Dynamics)")

## 5. Model Evaluation and Validation

Let's examine the quality of our model fit more closely.

In [ ]:
# Calculate residuals
predictions_for_resid = fitted_model.predict(t_data)
residuals = noisy_adoption - predictions_for_resid

# Plot residuals
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs fitted values
ax1.scatter(predictions_for_resid, residuals, alpha=0.6)
ax1.axhline(y=0, color='r', linestyle='--', linewidth=2)
ax1.set_xlabel('Fitted Values')
ax1.set_ylabel('Residuals')
ax1.set_title('Residuals vs Fitted Values')
ax1.grid(True, alpha=0.3)

# Q-Q plot for normality of residuals
from scipy import stats
stats.probplot(residuals, dist="norm", plot=ax2)
ax2.set_title('Q-Q Plot of Residuals')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate additional metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error

mse = mean_squared_error(noisy_adoption, predictions_for_resid)
mae = mean_absolute_error(noisy_adoption, predictions_for_resid)
rmse = np.sqrt(mse)

print(f"Model Evaluation Metrics:")
print(f"  R² Score: {r_squared:.4f}")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAE: {mae:.4f}")
print(f"  MSE: {mse:.4f}")

## Summary

This notebook has demonstrated:
1. Basic Bass model fitting to synthetic data
2. Incorporating covariates like marketing spend
3. Predicting adoption rates
4. The range of models available in the library
5. Model evaluation techniques

The innovate library provides a flexible and powerful framework for modeling innovation diffusion with various features for real-world applications.